In [6]:
from zfish._io import import16chFlt
from zfish.local_path import *
import os

import numpy as np
import matplotlib.pyplot as plt
from zfish.utils import *

code_id = "1001 - Range-based Shuffle Test"
loc = os.path.join(figpath, code_id)
os.makedirs(loc, exist_ok=True)

def reshuffle_test(save_dir, n_map=1, n_shuffle=1000):
    map_ids = [4, 5]
    exclude_prefix = 200
    
    with open(save_dir, 'rb') as f:
        trace = pickle.load(f)
    
    
    p_values_all = trace['p_values']
    ptp_all = trace['ptp']
    shuf_ptp_all = trace['shuf_ptp']
    
    n_map = p_values_all.shape[1]
    for n in range(n_map):
        p_values_all[:, n] = np.mean(shuf_ptp_all[:, n, :] >= ptp_all[:, n, np.newaxis], axis=1)
    
    trace['p_values'] = p_values_all
    with open(save_dir, 'wb') as f:
        pickle.dump(trace, f)
    print(f"{save_dir} reshuffled.")

dir_names = [
    r"D:\EnData\Light-sheet\10138\S3\trace.pkl",
    r"D:\EnData\Light-sheet\spatial preference vr1\10156\S2\trace.pkl",
    r"D:\EnData\Light-sheet\spatial preference vr1\10158\S2\trace.pkl",
    r"D:\EnData\Light-sheet\spatial preference vr1\10159\S2\trace.pkl",
    r"D:\EnData\Light-sheet\spatial preference vr1\10162\S2\trace.pkl",
    r"D:\EnData\Light-sheet\spatial preference vr1\10172\S2\trace.pkl",
]

for dir in dir_names:
    try:
        reshuffle_test(dir, n_map=1, n_shuffle=1000)
    except:
        pass

D:\EnData\Light-sheet\spatial preference vr1\10156\S2\trace.pkl reshuffled.
D:\EnData\Light-sheet\spatial preference vr1\10158\S2\trace.pkl reshuffled.
D:\EnData\Light-sheet\spatial preference vr1\10162\S2\trace.pkl reshuffled.
D:\EnData\Light-sheet\spatial preference vr1\10172\S2\trace.pkl reshuffled.


In [ ]:
def visualize(dir_name: str, threshold=0.05):
    with open(dir_name, 'rb') as f:
        trace = pickle.load(f)
        
    if 'p_values' not in trace.keys():
        print(f"{dir_name} does not have p_values.")
        return
        
    fish = trace['FishID']
    
    meanImg, rois_adjust = get_meanImg(trace)
    
    if trace['p_values'].ndim == 1:
        n_map = 1
    else:
        n_map = trace['p_values'].shape[1]
    
    if n_map == 1:
        significant_idx = np.where((trace['p_values'] < threshold)&(trace['snr'] >= 0.4))[0]
        fig, axes = plt.subplots(ncols=2, nrows=1, figsize=(8, 2))
        ax1 = Clear_Axes(axes[0])
        ax1.imshow(np.mean(meanImg, axis=2), cmap='gray', aspect='equal')
        ax1.scatter(
            rois_adjust[significant_idx, 0],
            rois_adjust[significant_idx, 1],
            s=1, c=trace['fir_sec_corr'][significant_idx, 0], cmap='rainbow', edgecolors=None, alpha=0.5, vmin=0
        )
        ax2 = Clear_Axes(axes[1])
        ax2.imshow(np.mean(meanImg, axis=0).T, cmap='gray', aspect='auto')
        ax2.scatter(
            rois_adjust[significant_idx, 0],
            rois_adjust[significant_idx, 2]+np.random.rand(significant_idx.shape[0])-0.5,
            c=trace['fir_sec_corr'][significant_idx, 0],
            s=1, cmap='rainbow', edgecolors=None, alpha=0.5, vmin=0
        )
        ax1.set_xlabel("Caudal → Rostro")
        ax1.set_ylabel("Lateral → Medial ← Lateral")
        ax2.set_xlabel("Caudal → Rostro")
        ax2.set_ylabel("Ventral → Dorsal")
    else:
        fig, axes = plt.subplots(ncols=2, nrows=n_map, figsize=(8, 2*n_map))
        for i in range(n_map):
            significant_idx = np.where((trace['p_values'][:, i] < threshold)&(trace['snr'] >= 0.4))[0]
            ax1 = Clear_Axes(axes[i, 0])
            ax1.imshow(np.mean(meanImg, axis=2), cmap='gray', aspect='equal')
            ax1.scatter(
                rois_adjust[significant_idx, 0],
                rois_adjust[significant_idx, 1],
                s=1, c=trace['fir_sec_corr'][significant_idx, i], cmap='rainbow', edgecolors=None, alpha=0.5, vmin=0
            )
            ax2 = Clear_Axes(axes[i, 1])
            ax2.imshow(np.mean(meanImg, axis=0).T, cmap='gray', aspect='auto')
            ax2.scatter(
                rois_adjust[significant_idx, 0],
                rois_adjust[significant_idx, 2]+np.random.rand(significant_idx.shape[0])-0.5,
                c=trace['fir_sec_corr'][significant_idx, i],
                s=1, cmap='rainbow', edgecolors=None, alpha=0.5, vmin=0
            )
            ax1.set_xlabel("Caudal → Rostro")
            ax1.set_ylabel(f"Linear Track {i+1}\nLateral → Medial ← Lateral")
            ax2.set_xlabel("Caudal → Rostro")
            ax2.set_ylabel("Ventral → Dorsal")
        
        
    plt.tight_layout()
    plt.suptitle(f"{fish} - {significant_idx.shape[0]} significant cells")
    plt.savefig(os.path.join(loc, f"Significant Cells Distribution [{fish}].png"), dpi=600)
    plt.savefig(os.path.join(loc, f"Significant Cells Distribution [{fish}].svg"), dpi=600)
    plt.show()
    
for i, dir in enumerate(dir_names):
    if i <= 3:
        continue
    visualize(dir)